In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

import nengo
import nengo_dl

from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical


In [ ]:
# Load MNIST data
print("Loading MNIST dataset...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize images to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Reshape for CNN: (samples, height, width, channels)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# One-hot encode labels
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

print(f"Training samples: {len(x_train)}")
print(f"Test samples: {len(x_test)}")
print(f"Image shape: {x_train[0].shape}")

Loading MNIST dataset...
Training samples: 60000
Test samples: 10000
Image shape: (28, 28, 1)


In [ ]:
# Build LeNet-5 model
print("Building LeNet-5 ANN model...")

ann_model = tf.keras.Sequential([
    # C1: Convolutional layer
    tf.keras.layers.Conv2D(6, (5,5), activation='tanh', input_shape=(28,28,1)),
    # S2: Pooling layer
    tf.keras.layers.AveragePooling2D((2,2)),
    # C3: Convolutional layer
    tf.keras.layers.Conv2D(6, (5,5), activation='tanh'),
    # S4: Pooling layer
    tf.keras.layers.AveragePooling2D((2,2)),
    # Flatten
    tf.keras.layers.Flatten(),
    # FC5: Fully connected
    tf.keras.layers.Dense(120, activation='tanh'),
    # FC6: Fully connected
    tf.keras.layers.Dense(84, activation='tanh'),
    # Output layer
    tf.keras.layers.Dense(10, activation='softmax')
])

ann_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

print("\nModel Architecture:")
ann_model.summary()


Building LeNet-5 ANN model...

Model Architecture:
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 24, 24, 6)         156       
                                                                 
 average_pooling2d (AverageP  (None, 12, 12, 6)        0         
 ooling2D)                                                       
                                                                 
 conv2d_1 (Conv2D)           (None, 8, 8, 6)           906       
                                                                 
 average_pooling2d_1 (Averag  (None, 4, 4, 6)          0         
 ePooling2D)                                                     
                                                                 
 flatten (Flatten)           (None, 96)                0         
                                                                 
 dens

In [ ]:
# Train the ANN
print("Training ANN...")

history = ann_model.fit(
    x_train, y_train,

    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)



Training ANN...
Epoch 1/10
422/422 [==============================] - 6s 12ms/step - loss: 0.4297 - accuracy: 0.8780 - val_loss: 0.1605 - val_accuracy: 0.9560
Epoch 2/10
422/422 [==============================] - 5s 12ms/step - loss: 0.1582 - accuracy: 0.9518 - val_loss: 0.1024 - val_accuracy: 0.9717
Epoch 3/10
422/422 [==============================] - 5s 11ms/step - loss: 0.1080 - accuracy: 0.9673 - val_loss: 0.0890 - val_accuracy: 0.9725
Epoch 4/10
422/422 [==============================] - 5s 11ms/step - loss: 0.0833 - accuracy: 0.9742 - val_loss: 0.0723 - val_accuracy: 0.9775
Epoch 5/10
422/422 [==============================] - 5s 12ms/step - loss: 0.0693 - accuracy: 0.9787 - val_loss: 0.0664 - val_accuracy: 0.9803
Epoch 6/10
422/422 [==============================] - 5s 12ms/step - loss: 0.0592 - accuracy: 0.9820 - val_loss: 0.0598 - val_accuracy: 0.9820
Epoch 7/10
422/422 [==============================] - 5s 12ms/step - loss: 0.0524 - accuracy: 0.9837 - val_loss: 0.0543 - val_

In [ ]:
# Evaluate ANN
ann_loss, ann_accuracy = ann_model.evaluate(x_test, y_test, verbose=0)
print(f"\nANN Test Accuracy: {ann_accuracy*100:.2f}%")
print(f"ANN Test Loss: {ann_loss:.4f}")


ANN Test Accuracy: 98.41%
ANN Test Loss: 0.0520


In [ ]:
converter = nengo_dl.Converter(
    ann_model,
    swap_activations={tf.keras.activations.relu: nengo.LIF}
)
 #Converts the trained ANN model into an equivalent SNN and transfers its learned weights.

snn_network = converter.net  #Retrieves the converted spiking neural network from the converter.

output_probe = list(converter.outputs.values())[-1]  #Gets the probe that records the activity of the final output layer of the SNN.

print("\nANN successfully converted to SNN")




ANN successfully converted to SNN


c:\Users\Hari Nisanth S\AppData\Local\Programs\Python\Python39\lib\site-packages\nengo_dl\converter.py:140: UserWarning: swap_activations contained {<function relu at 0x000001538C1851F0>}, but there were no layers in the model with that activation type
  warnings.warn(


In [ ]:
with nengo_dl.Simulator(snn_network, minibatch_size=128) as sim:
    print("Weights ready for SNN evaluation")

    timesteps = 200 # Number of time steps to run the SNN simulation for each input sample ie. how long the SNN will process each image.
    n_test = 1000 # Number of test samples to evaluate the SNN on. We use a subset of the test set to keep simulation time reasonable.

    # 1. Prepare data
    x_test_snn = np.tile(
        x_test[:n_test].reshape(n_test, -1)[:, None, :],
        (1, timesteps, 1)
    ) # Reshape and tile test data to match SNN input format: (samples, timesteps, features)

    # 2. Run simulation
    print(f"Running SNN simulation...")
    sim_data = sim.predict(x_test_snn) # Run the SNN simulation on the prepared test data and collect the output from the specified probe.

    # 3. Get predictions
    snn_outputs = sim_data[output_probe]
    predictions = np.argmax(np.mean(snn_outputs, axis=1), axis=-1) # Average the SNN output over time steps and take the argmax to get predicted class labels.

    # --- THE FIX STARTS HERE ---
    # Only take the number of true labels that the simulator actually processed
    num_predictions = predictions.shape[0]
    true_labels = np.argmax(y_test[:num_predictions], axis=-1)

    # Now the shapes will match: (896,) == (896,)
    snn_accuracy = np.mean(predictions == true_labels)
    # --- THE FIX ENDS HERE ---

    print("\n" + "="*60)
    print("COMPARISON RESULTS:")
    print("="*60)
    print(f"ANN Accuracy:  {ann_accuracy*100:.2f}%")
    print(f"SNN Accuracy:  {snn_accuracy*100:.2f}% (on {num_predictions} samples)")
    print("="*60)

|                     Building network (0%)                    | ETA:  --:--:--
Build finished in 0:00:00
|#                         Optimizing graph                           | 0:00:00
|#             Optimizing graph: operator simplificaton               | 0:00:00
Optimizing graph: operator simplificaton finished in 0:00:00
|#                Optimizing graph: merging operators                 | 0:00:00
Optimizing graph: merging operators finished in 0:00:00
|#                Optimizing graph: ordering signals                  | 0:00:00
Optimizing graph: ordering signals finished in 0:00:00
|#                Optimizing graph: creating signals                  | 0:00:00
Optimizing graph: creating signals finished in 0:00:00
Optimization finished in 0:00:00
|#                        Constructing graph                          | 0:00:00
| #                       Constructing graph                          | 0:00:00
|           Constructing graph: pre-build stage (0%)           | ETA:  --:

In [ ]:
with nengo_dl.Simulator(snn_network, minibatch_size=1) as sim:

    test_idx = 7956 % len(x_test)  # Ensure valid index
    test_image = x_test[test_idx:test_idx+1]

    # Prepare for SNN
    test_image_snn = np.tile(
        test_image.reshape(1, -1)[:, None, :],
        (1, timesteps, 1)
    )

    # Get ANN prediction
    ann_pred = ann_model.predict(test_image, verbose=0)[0]

    # Get SNN prediction
    snn_pred_raw = sim.predict(test_image_snn)[output_probe]
    snn_pred = np.mean(snn_pred_raw, axis=1)[0]

    # Normalize SNN output
    snn_pred_norm = np.exp(snn_pred) / np.sum(np.exp(snn_pred))

    # Get true label
    true_label = np.argmax(y_test[test_idx])

    print(f"\nTest Image Index: {test_idx}")
    print(f"True Label: {true_label}")
    print(f"ANN Prediction: {np.argmax(ann_pred)} (confidence: {np.max(ann_pred):.3f})")
    print(f"SNN Prediction: {np.argmax(snn_pred)} (confidence: {np.max(snn_pred_norm):.3f})")


|                     Building network (0%)                    | ETA:  --:--:--
Build finished in 0:00:00
|#                         Optimizing graph                           | 0:00:00
|#             Optimizing graph: operator simplificaton               | 0:00:00
Optimizing graph: operator simplificaton finished in 0:00:00
|#                Optimizing graph: merging operators                 | 0:00:00
Optimizing graph: merging operators finished in 0:00:00
|#                Optimizing graph: ordering signals                  | 0:00:00
Optimizing graph: ordering signals finished in 0:00:00
|#                Optimizing graph: creating signals                  | 0:00:00
Optimizing graph: creating signals finished in 0:00:00
Optimization finished in 0:00:00
|#                        Constructing graph                          | 0:00:00
| #                       Constructing graph                          | 0:00:00
|           Constructing graph: pre-build stage (0%)           | ETA:  --: